The code in this notebook was inspired and partially copied/derived from these articles:

https://towardsdatascience.com/lime-vs-shap-which-is-better-for-explaining-machine-learning-models-d68d8290bb16

LIME vs. SHAP: Which is Better for Explaining Machine Learning Models?
Two of the most popular Explainers compared
Dario Radečić, Dec 14, 2020

https://towardsdatascience.com/squeezing-more-out-of-lime-with-python-28f46f74ca8e
Squeezing More out of LIME with Python
How to create global aggregations of LIME weights
Conor O'Sullivan, May 4 2022

However many of the features do not work for MLP.

In [0]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [0]:
%pip install -q numpy=1.26.4 pandas scikit-learn lime shap
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Download breast cancer dataset as a DataFrame
raw_data = load_breast_cancer(as_frame=True)
breast_cancer_data = raw_data.frame

type(breast_cancer_data)

# Do not truncate columns when displaying null counts and value counts
pd.set_option('display.max_columns', None)

# Check for missing values and class distribution
print(breast_cancer_data.isnull().sum().sum())
print(breast_cancer_data['target'].value_counts())

# Display the first few rows of the dataset
breast_cancer_data.head()

In [0]:
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

X = breast_cancer_data.drop('target', axis=1)
y = breast_cancer_data['target']

cormat = X.corr()

plt.figure(figsize=(12,10))
sns.heatmap(cormat, square=True, center=0)

In [0]:
# Split data set into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Set the random number generation seed so that results can be duplicated
np.random.seed(12345)

# Fit a Neural Network model with the training data
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(30,30,30), activation='relu', solver='adam', max_iter=20000)
mlp.fit(X_train.values,y_train)

In [0]:
from sklearn.metrics import classification_report

# Provide quality of fit metrics with the test dataset
y_pred = mlp.predict(X_test)
target_names = set(raw_data['target_names'])
print(X_test.columns)
print(classification_report(y_test, y_pred, target_names=target_names))

In [0]:
import lime
from lime import lime_tabular

test_1 = X_test.iloc[14] # Instance explained using LIME
print(test_1, y_pred[14])

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    class_names=['malignant', 'benign'],
    mode='classification'
)

# Explain one instance of the data with lime
lime_exp = lime_explainer.explain_instance(
    data_row=test_1,
    predict_fn=mlp.predict_proba
)
lime_exp.show_in_notebook(show_table=True)

In [0]:
weights = []

# Function to get weights from LIME explanation object
def return_weights(ep):
    exp_list = exp.as_map()[1] # benign
    exp_list = sorted(exp_list, key=lambda x: x[0])
    exp_weight = [x[1] for x in exp_list]
    return exp_weight

print(X_test.columns)
print(len(X_test))

# Iterate over the rows in feature matrix and collect the LIME weights

for x in range(min(len(X_test),25)):
    #Get explanation
    exp = lime_explainer.explain_instance(X_test.iloc[x],
                                 mlp.predict_proba, num_features = len(X_test.columns))
    #Get weights
    exp_weight = return_weights(exp)
    weights.append(exp_weight)

# Create DataFrame of the LIME weights
lime_weights = pd.DataFrame(data=weights,columns=X_test.columns)

# Get absolute value of the mean of LIME weights
abs_mean = lime_weights.abs().mean(axis=0)
abs_mean = pd.DataFrame(data={'feature':abs_mean.index, 'abs_mean':abs_mean})
abs_mean = abs_mean.sort_values('abs_mean')

# Plot abs mean LIME weights
fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(8,8))

y_ticks = range(len(abs_mean))
y_labels = abs_mean.feature
plt.barh(y=y_ticks,width=abs_mean.abs_mean)

plt.yticks(ticks=y_ticks,labels=y_labels,size= 15)
plt.title('')
plt.ylabel('')
plt.xlabel('Mean |Weight|',size=20)

In [0]:
# SHAP explainer for class 0 (malignant)

shap_explainer = shap.KernelExplainer(mlp.predict_proba, X_train)
shap_values = shap_explainer.shap_values(X_test.iloc[14, :])

shap.force_plot(
    shap_explainer.expected_value[0],
    shap_values[:, 0],
    X_test.iloc[14, :],
    matplotlib=True
)
plt.show()

In [0]:
# SHAP global explanation for class 1 (benign)
shap_explanation = shap.Explanation(
    values=shap_vals[:, :, 1],
    base_values=shap_explainer.expected_value[1],
    data=X_test.values,
    feature_names=feat_names
)
shap.plots.heatmap(shap_explanation)

In [0]:
# Create a summary plot of the influence of all input variables on the test dataset
shap_explainer = shap.KernelExplainer(mlp.predict_proba, X_train)

shap_vals = shap_explainer.shap_values(X_test)

feat_names = list(X_test.columns)
shap.plots.violin(shap_vals[:,:,0], features=X_test, feature_names=feat_names, plot_type="layered_violin") # label: malignant
shap.plots.violin(shap_vals[:,:,1], features=X_test, feature_names=feat_names, plot_type="layered_violin") # label: benign